---

## 🧩 When to Use `@dynamic_prompt`

| Scenario | What to read | Example |
|---|---|---|
| 🌍 Language/locale | `request.runtime.context` | Respond in user's language |
| 👤 User role | `request.runtime.context` | Admin vs customer instructions |
| 📊 Conversation phase | `request.messages` | Onboarding → solving → resolution |
| 🕐 Time-based | `datetime.now()` | "It's after hours, suggest FAQ" |
| 🔧 Tool-aware | `request.tools` | "You have access to: X, Y, Z" |
| 🗂️ State-aware | `request.runtime.state` | "User already provided X, don't ask again" |

> 💡 **Pro tip**: you can combine `@dynamic_prompt` with other middleware. For example, use `@dynamic_prompt` for the system prompt AND `SummarizationMiddleware` for message management — they don't conflict.

# ⚙️ Dynamic Prompts with `@dynamic_prompt`

## 🤔 The Problem with Static Prompts

When you create an agent with `system_prompt="You are a helpful assistant."`, that prompt is **fixed** — every user, every turn, every context gets the same instructions. But in real apps you need the prompt to change based on:

- 🌍 The user's language or locale
- 👤 The user's role (admin vs customer vs guest)
- 🕐 The time of day or business hours
- 📊 What the agent has already learned (conversation state)
- 🔧 Which tools are available in this session

## ✅ The Solution: `@dynamic_prompt`

`@dynamic_prompt` is a config-style middleware that **generates the system prompt at runtime** based on the current `ModelRequest`. It replaces the static `system_prompt=` parameter.

```python
@dynamic_prompt
def my_prompt(request: ModelRequest) -> str:
    # Read anything: context, state, messages, tools
    role = request.runtime.context.user_role
    return f"You are a {role}-level assistant."
```

### What `ModelRequest` gives you access to

| Field | What it contains |
|---|---|
| `request.messages` | Current conversation history |
| `request.runtime.context` | Read-only context (from `context=` at invoke) |
| `request.runtime.state` | Mutable agent state |
| `request.tools` | Tools available to the agent |

> ⚠️ **Don't set both** `system_prompt=` and `@dynamic_prompt` — the dynamic one wins.

---

## 📝 Example 1: Language-Based Prompt

The simplest use case — change the response language based on user context.

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from dataclasses import dataclass
from langchain.agents.middleware import dynamic_prompt, ModelRequest


@dataclass
class LanguageContext:
    user_language: str = "English"


@dynamic_prompt
def user_language_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_language = request.runtime.context.user_language
    base_prompt = "You are a helpful assistant."

    if user_language != "English":
        return f"{base_prompt} only respond in {user_language}."
    elif user_language == "English":
        return base_prompt

In [3]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    context_schema=LanguageContext,
    middleware=[user_language_prompt]
)

In [4]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Irish")
)

print(response["messages"][-1].content)

Dia dhuit! Tá mé go maith, go raibh maith agat. Conas atá tú?


In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Spanish")
)

print(response["messages"][-1].content)

¡Hola! Estoy bien, gracias. ¿Y tú? ¿En qué puedo ayudarte hoy?


In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="French")
)

print(response["messages"][-1].content)

Bonjour ! Je vais très bien, merci. Et toi, comment vas-tu ?


---

## 📝 Example 2: Role-Based Prompt

Different users get different instructions. An admin sees technical details; a customer gets friendly support language.

In [ ]:
@dataclass
class UserContext:
    user_role: str = "customer"   # "admin", "customer", or "guest"
    user_name: str = "Friend"


@dynamic_prompt
def role_based_prompt(request: ModelRequest) -> str:
    """Tailor the system prompt based on user role."""
    ctx = request.runtime.context
    role = ctx.user_role
    name = ctx.user_name

    if role == "admin":
        return (
            f"You are a technical assistant for admin {name}. "
            "Be precise, include error codes, stack traces, and config details. "
            "Assume the user is an expert."
        )
    elif role == "customer":
        return (
            f"You are a friendly support agent helping {name}. "
            "Use simple language, avoid jargon, and be empathetic. "
            "Always offer to escalate if the issue is complex."
        )
    else:  # guest
        return (
            "You are a helpful assistant. The user is not logged in. "
            "Provide general information only. Do not access account data."
        )


agent_role = create_agent(
    model="gpt-5-nano",
    context_schema=UserContext,
    middleware=[role_based_prompt],
)

In [ ]:
# Admin gets technical, detailed response
response = agent_role.invoke(
    {"messages": [HumanMessage(content="The server is returning 502 errors")]},
    context=UserContext(user_role="admin", user_name="Alice"),
)
print("🔧 Admin response:")
print(response["messages"][-1].content)

In [ ]:
# Customer gets friendly, simple response
response = agent_role.invoke(
    {"messages": [HumanMessage(content="The website isn't working")]},
    context=UserContext(user_role="customer", user_name="Bob"),
)
print("😊 Customer response:")
print(response["messages"][-1].content)

---

## 📝 Example 3: State-Aware Prompt

The prompt changes based on **what the agent has already learned** during the conversation. This uses `request.messages` to inspect the conversation history.

> 💡 This is powerful for multi-step workflows — the agent's personality/instructions evolve as the conversation progresses.

In [ ]:
from langchain.messages import AIMessage


@dynamic_prompt
def conversation_phase_prompt(request: ModelRequest) -> str:
    """Change the prompt based on how far along the conversation is."""
    msg_count = len(request.messages)

    if msg_count <= 2:
        return (
            "You are a friendly onboarding assistant. "
            "Ask the user's name and what they need help with. "
            "Be warm and welcoming."
        )
    elif msg_count <= 6:
        return (
            "You are now in problem-solving mode. "
            "The user has introduced themselves. "
            "Focus on understanding their issue. Ask clarifying questions."
        )
    else:
        return (
            "You are in resolution mode. "
            "You've gathered enough context. "
            "Provide a clear, actionable solution. Be concise."
        )


agent_phased = create_agent(
    model="gpt-5-nano",
    middleware=[conversation_phase_prompt],
)

In [ ]:
# Phase 1: Onboarding (≤2 messages)
response = agent_phased.invoke(
    {"messages": [HumanMessage(content="Hi there")]}
)
print("Phase 1 (onboarding):")
print(response["messages"][-1].content)

In [ ]:
# Phase 2: Problem-solving (3-6 messages)
response = agent_phased.invoke(
    {"messages": [
        HumanMessage(content="Hi there"),
        AIMessage(content="Welcome! What's your name?"),
        HumanMessage(content="I'm Alice. My app keeps crashing."),
    ]}
)
print("Phase 2 (problem-solving):")
print(response["messages"][-1].content)

In [ ]:
# Phase 3: Resolution (7+ messages)
response = agent_phased.invoke(
    {"messages": [
        HumanMessage(content="Hi there"),
        AIMessage(content="Welcome! What's your name?"),
        HumanMessage(content="I'm Alice. My app keeps crashing."),
        AIMessage(content="What device and OS are you using?"),
        HumanMessage(content="iPhone 15, iOS 18."),
        AIMessage(content="When does it crash? On launch or during use?"),
        HumanMessage(content="It crashes when I open the camera."),
    ]}
)
print("Phase 3 (resolution):")
print(response["messages"][-1].content)